In [1]:
# Cell 1: Imports and setup
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import requests
import io
from kloppy import impect
from kloppy.utils import github_resolve_raw_data_url
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports successful")



All imports successful


In [2]:
# Cell 2: Setup paths
from pathlib import Path

# Go up one level from Notebooks/ to project root
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Create if needed
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")

Project root: /Users/tanishbhilare/Desktop/SoccerImpectHackathon
Data directory: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data


### Load Event Dataset

Load the complete event dataset with team-aware zone classifications from Notebook 1 to enable validation analysis and metric calculations.

In [3]:
# Cell 3: Load base event data with team-aware zones

print("Loading base event data...")

# Load events with zones from Notebook 1
all_events = pl.read_parquet(PROCESSED_DIR / "all_matches_with_zones.parquet")

print(f"Loaded {len(all_events):,} events")
print(f"  Unique players: {all_events['player_id'].n_unique()}")
print(f"  Unique matches: {all_events['match_id'].n_unique()}")

# Display available columns
print(f"\nAvailable columns ({len(all_events.columns)}):")
print(all_events.columns)

# Display sample events
print(f"\nSample events:")
print(all_events.select([
    'event_type', 'player_id', 'zone', 'result', 'success',
    'coordinates_x', 'end_coordinates_x'
]).sample(5))

print("Base event data loaded")


Loading base event data...
Loaded 962,990 events
  Unique players: 494
  Unique matches: 306

Available columns (28):
['event_id', 'event_type', 'period_id', 'timestamp', 'end_timestamp', 'ball_state', 'ball_owning_team', 'team_id', 'player_id', 'coordinates_x', 'coordinates_y', 'end_coordinates_x', 'end_coordinates_y', 'receiver_player_id', 'body_part_type', 'set_piece_type', 'result', 'success', 'duel_type', 'is_under_pressure', 'pass_type', 'goalkeeper_type', 'match_id', 'home_squad_id', 'away_squad_id', 'zone', 'end_zone', 'card_type']

Sample events:
shape: (5, 7)
┌────────────┬───────────┬─────────────────┬────────────┬─────────┬───────────────┬────────────────┐
│ event_type ┆ player_id ┆ zone            ┆ result     ┆ success ┆ coordinates_x ┆ end_coordinate │
│ ---        ┆ ---       ┆ ---             ┆ ---        ┆ ---     ┆ ---           ┆ s_x            │
│ str        ┆ str       ┆ str             ┆ str        ┆ bool    ┆ f64           ┆ ---            │
│            ┆      

### Helper Functions for Event Classification

Define utility functions to identify special event characteristics (progressive actions, long passes, box entries) used in granular metric calculations.

In [4]:
# Cell 4: Helper functions to identify special event types

print("Defining helper functions for event classification")

# Utility function for distance calculation
def calculate_distance(start_x, start_y, end_x, end_y):
    """
    Calculate Euclidean distance between two points.
    
    Args:
        start_x, start_y: Starting coordinates
        end_x, end_y: Ending coordinates
    
    Returns:
        float: Distance in meters, or None if coordinates missing
    """
    if None in [start_x, start_y, end_x, end_y]:
        return None
    
    distance = np.sqrt((end_x - start_x)**2 + (end_y - start_y)**2)
    return distance


# Attacking event helpers
def is_progressive_pass(row):
    """
    Check if pass is progressive (advances ball ≥10m toward opponent goal).
    
    Logic: Progressive if (end_x - start_x) ≥ 10 meters
    Threshold: 10m is industry standard (StatsBomb, Opta)
    
    Returns:
        bool: True if progressive, False otherwise
    """
    start_x = row.get('coordinates_x')
    end_x = row.get('end_coordinates_x')
    
    if start_x is None or end_x is None:
        return False
    
    forward_distance = end_x - start_x
    return forward_distance >= 10.0


def is_progressive_carry(row):
    """
    Check if carry/dribble is progressive (advances ball ≥10m forward).
    
    Logic: Same as progressive pass, applied to carries
    
    Returns:
        bool: True if progressive, False otherwise
    """
    start_x = row.get('coordinates_x')
    end_x = row.get('end_coordinates_x')
    
    if start_x is None or end_x is None:
        return False
    
    forward_distance = end_x - start_x
    return forward_distance >= 10.0


def is_long_pass(row):
    """
    Check if pass is long-range (total distance ≥30m).
    
    Logic: Calculate Euclidean distance using Pythagorean theorem
    Threshold: 30m approximates switching play or long ball
    
    Returns:
        bool: True if long pass, False otherwise
    """
    start_x = row.get('coordinates_x')
    start_y = row.get('coordinates_y')
    end_x = row.get('end_coordinates_x')
    end_y = row.get('end_coordinates_y')
    
    distance = calculate_distance(start_x, start_y, end_x, end_y)
    
    if distance is None:
        return False
    
    return distance >= 30.0


def is_into_penalty_area(row):
    """
    Check if pass/carry ends in opponent's penalty area.
    
    Logic: Penalty area in secondspectrum coordinates:
           - x > 35.5 (near opponent goal, 16.5m from goal line)
           - -9.15 < y < 9.15 (width of penalty box)
    
    Returns:
        bool: True if ends in penalty area, False otherwise
    """
    end_x = row.get('end_coordinates_x')
    end_y = row.get('end_coordinates_y')
    
    if end_x is None or end_y is None:
        return False
    
    in_box_x = end_x > 35.5
    in_box_y = -9.15 < end_y < 9.15
    
    return in_box_x and in_box_y


def is_into_final_third(row):
    """
    Check if pass/carry crosses into attacking third.
    
    Logic: Must START outside attacking third and END inside it
    
    Returns:
        bool: True if crosses into final third, False otherwise
    """
    start_zone = row.get('zone')
    end_zone = row.get('end_zone')
    
    if start_zone is None or end_zone is None:
        return False
    
    crosses_into_attack = (
        start_zone != 'attacking_third' and 
        end_zone == 'attacking_third'
    )
    
    return crosses_into_attack


def is_carry_into_box(row):
    """
    Check if carry/dribble ends in opponent's penalty area.
    
    High value action: beating defenders with the ball into dangerous area
    
    Returns:
        bool: True if dribble into box, False otherwise
    """
    return is_into_penalty_area(row)


# Defensive event helpers
def is_in_own_penalty_area(row):
    """
    Check if event occurred in own penalty area (critical defensive zone).
    
    Logic: Own penalty area in secondspectrum:
           - x < -35.5 (near own goal)
           - -9.15 < y < 9.15 (width of box)
    
    Returns:
        bool: True if in own penalty area, False otherwise
    """
    x = row.get('coordinates_x')
    y = row.get('coordinates_y')
    
    if x is None or y is None:
        return False
    
    in_own_box_x = x < -35.5
    in_box_y = -9.15 < y < 9.15
    
    return in_own_box_x and in_box_y


# Display summary
print("\nHelper functions defined:")
print("  Attacking: progressive_pass, progressive_carry, long_pass, into_penalty_area, into_final_third, carry_into_box")
print("  Defensive: in_own_penalty_area")
print("  Utility: calculate_distance")

# Test functions on sample events
print("\nTesting helper functions on sample events:")

test_events = all_events.filter(
    pl.col('event_type').is_in(['PASS', 'CARRY'])
).sample(5)

for i, row_dict in enumerate(test_events.to_dicts(), 1):
    print(f"\n  Event {i} ({row_dict['event_type']}):")
    
    start_x = row_dict.get('coordinates_x')
    end_x = row_dict.get('end_coordinates_x')
    
    if start_x is not None and end_x is not None:
        print(f"    Start: ({start_x:.1f}, {row_dict.get('coordinates_y', 0):.1f}) -> End: ({end_x:.1f}, {row_dict.get('end_coordinates_y', 0):.1f})")
        print(f"    Zone: {row_dict.get('zone')} -> {row_dict.get('end_zone')}")
        print(f"    Progressive: {is_progressive_pass(row_dict) if row_dict['event_type'] == 'PASS' else is_progressive_carry(row_dict)}")
        print(f"    Into box: {is_into_penalty_area(row_dict)}")

print("\nHelper function validation complete")

Defining helper functions for event classification

Helper functions defined:
  Attacking: progressive_pass, progressive_carry, long_pass, into_penalty_area, into_final_third, carry_into_box
  Defensive: in_own_penalty_area
  Utility: calculate_distance

Testing helper functions on sample events:

  Event 1 (CARRY):
    Start: (-8.6, -32.5) -> End: (-8.3, -32.0)
    Zone: middle_third -> middle_third
    Progressive: False
    Into box: False

  Event 2 (PASS):

  Event 3 (PASS):
    Start: (36.3, -16.4) -> End: (43.0, -23.3)
    Zone: attacking_third -> attacking_third
    Progressive: False
    Into box: False

  Event 4 (CARRY):
    Start: (-33.4, 16.5) -> End: (-23.7, 21.3)
    Zone: defensive_third -> defensive_third
    Progressive: False
    Into box: False

  Event 5 (PASS):
    Start: (27.4, 33.0) -> End: (19.3, 25.0)
    Zone: defensive_third -> defensive_third
    Progressive: False
    Into box: False

Helper function validation complete


In [5]:
# Cell 5: Calculate granular metric points for all events

print("Calculating metric points for all events")
print("Processing 962,990 events (estimated time: 30-45 seconds)")

import time
start_time = time.time()

# Convert to list of dicts for iteration
events_list = all_events.to_dicts()

# Initialize point lists for 8 metrics
finishing_points = []
chance_creation_points = []
ball_progression_points = []
dribbling_points = []
ball_winning_points = []
defensive_actions_points = []
passing_accuracy_points = []
long_passing_points = []

# Point calculation functions
def calc_finishing_points(row):
    """Calculate finishing metric points"""
    if row['event_type'] != 'SHOT':
        return 0.0
    
    result = row.get('result')
    if result == 'GOAL':
        return 10.0
    elif result == 'SAVED':
        return 2.0
    elif result == 'BLOCKED':
        return 0.5
    elif result == 'OFF_TARGET':
        return 0.3
    return 0.0


def calc_chance_creation_points(row):
    """Calculate chance creation metric points"""
    if row['event_type'] != 'PASS':
        return 0.0
    
    pass_type = row.get('pass_type')
    
    if pass_type == 'SHOT_ASSIST':
        return 10.0
    
    if row.get('result') == 'COMPLETE' and is_into_penalty_area(row):
        return 3.0
    
    return 0.0


def calc_ball_progression_points(row):
    """Calculate ball progression metric points"""
    points = 0.0
    
    if row['event_type'] == 'PASS' and row.get('result') == 'COMPLETE':
        if is_progressive_pass(row):
            points += 2.0
        elif is_into_final_third(row):
            points += 1.5
    
    elif row['event_type'] == 'CARRY' and row.get('result') == 'COMPLETE':
        if is_progressive_carry(row):
            points += 2.5
    
    return points


def calc_dribbling_points(row):
    """Calculate dribbling metric points"""
    points = 0.0
    
    if row['event_type'] == 'CARRY' and row.get('result') == 'COMPLETE':
        if is_carry_into_box(row):
            points += 3.0
        elif is_progressive_carry(row):
            points += 1.5
    
    elif row['event_type'] == 'DUEL' and row.get('result') == 'WON':
        if row.get('zone') == 'attacking_third':
            points += 1.0
    
    return points


def calc_ball_winning_points(row):
    """Calculate ball winning metric points"""
    points = 0.0
    zone = row.get('zone')
    
    if row['event_type'] == 'DUEL' and row.get('result') == 'WON':
        if zone == 'defensive_third':
            points += 3.0
        elif zone == 'middle_third':
            points += 2.0
        elif zone == 'attacking_third':
            points += 1.0
    
    elif row['event_type'] == 'INTERCEPTION':
        if zone == 'defensive_third':
            points += 2.5
        elif zone == 'middle_third':
            points += 2.0
        elif zone == 'attacking_third':
            points += 1.5
    
    return points


def calc_defensive_actions_points(row):
    """Calculate defensive actions metric points"""
    points = 0.0
    zone = row.get('zone')
    
    if row['event_type'] == 'CLEARANCE':
        if is_in_own_penalty_area(row):
            points += 3.0
        elif zone == 'defensive_third':
            points += 2.0
        elif zone == 'middle_third':
            points += 1.0
    
    elif row['event_type'] == 'RECOVERY':
        if zone == 'defensive_third':
            points += 1.5
        elif zone == 'middle_third':
            points += 1.0
        elif zone == 'attacking_third':
            points += 0.8
    
    return points


def calc_passing_accuracy_points(row):
    """Calculate passing accuracy metric points"""
    if row['event_type'] != 'PASS':
        return 0.0
    
    result = row.get('result')
    
    if result == 'COMPLETE':
        if row.get('is_under_pressure', False):
            return 1.0
        return 0.5
    elif result == 'INCOMPLETE':
        return -0.3
    
    return 0.0


def calc_long_passing_points(row):
    """Calculate long passing metric points"""
    if row['event_type'] != 'PASS':
        return 0.0
    
    if is_long_pass(row):
        if row.get('result') == 'COMPLETE':
            return 2.0
        elif row.get('result') == 'INCOMPLETE':
            return -0.5
    
    return 0.0


# Calculate points for all events
for i, row in enumerate(events_list):
    finishing_points.append(calc_finishing_points(row))
    chance_creation_points.append(calc_chance_creation_points(row))
    ball_progression_points.append(calc_ball_progression_points(row))
    dribbling_points.append(calc_dribbling_points(row))
    ball_winning_points.append(calc_ball_winning_points(row))
    defensive_actions_points.append(calc_defensive_actions_points(row))
    passing_accuracy_points.append(calc_passing_accuracy_points(row))
    long_passing_points.append(calc_long_passing_points(row))
    
    # Progress indicator every 200k events
    if (i + 1) % 200000 == 0:
        elapsed = time.time() - start_time
        progress = (i + 1) / len(events_list)
        remaining = (elapsed / progress) * (1 - progress)
        print(f"  Processed {i+1:,}/{len(events_list):,} ({progress:.1%}) | "
              f"Elapsed: {elapsed:.1f}s | Remaining: {remaining:.1f}s")

# Add points as new columns
all_events = all_events.with_columns([
    pl.Series('finishing_points', finishing_points),
    pl.Series('chance_creation_points', chance_creation_points),
    pl.Series('ball_progression_points', ball_progression_points),
    pl.Series('dribbling_points', dribbling_points),
    pl.Series('ball_winning_points', ball_winning_points),
    pl.Series('defensive_actions_points', defensive_actions_points),
    pl.Series('passing_accuracy_points', passing_accuracy_points),
    pl.Series('long_passing_points', long_passing_points),
])

elapsed_total = time.time() - start_time
print(f"\nCalculated all metric points in {elapsed_total:.1f} seconds")

# Display summary statistics
print("\nMetric points summary:")

metric_cols = [
    'finishing_points', 'chance_creation_points', 'ball_progression_points',
    'dribbling_points', 'ball_winning_points', 'defensive_actions_points',
    'passing_accuracy_points', 'long_passing_points'
]

for metric in metric_cols:
    total = all_events[metric].sum()
    positive = (all_events[metric] > 0).sum()
    negative = (all_events[metric] < 0).sum()
    print(f"  {metric:30s}: Total={total:>9,.0f} | Positive={positive:>7,} | Negative={negative:>6,}")

# Sample events with points
print("\nSample events with calculated points:")
print(all_events.select([
    'event_type', 'zone', 'result',
    'finishing_points', 'chance_creation_points', 'ball_progression_points',
    'ball_winning_points'
]).sample(5))

print("\nMetric calculation complete")

Calculating metric points for all events
Processing 962,990 events (estimated time: 30-45 seconds)
  Processed 200,000/962,990 (20.8%) | Elapsed: 3.8s | Remaining: 14.3s
  Processed 400,000/962,990 (41.5%) | Elapsed: 4.1s | Remaining: 5.7s
  Processed 600,000/962,990 (62.3%) | Elapsed: 4.5s | Remaining: 2.7s
  Processed 800,000/962,990 (83.1%) | Elapsed: 4.8s | Remaining: 1.0s

Calculated all metric points in 5.2 seconds

Metric points summary:
  finishing_points              : Total=   14,965 | Positive=  8,069 | Negative=     0
  chance_creation_points        : Total=   34,328 | Positive=  4,802 | Negative=     0
  ball_progression_points       : Total=  166,494 | Positive= 81,028 | Negative=     0
  dribbling_points              : Total=   36,614 | Positive= 24,581 | Negative=     0
  ball_winning_points           : Total=   68,598 | Positive= 34,929 | Negative=     0
  defensive_actions_points      : Total=   66,056 | Positive= 57,036 | Negative=     0
  passing_accuracy_points    

### Extract Player Names

Load player names and jersey numbers from all 306 match metadata files, caching results for efficient reuse in subsequent runs.

In [6]:
# Cell 6: Extract complete player names from match metadata

print("Loading complete player names from match metadata")

# Load matches metadata
matches = pl.read_parquet(PROCESSED_DIR / "matches_metadata.parquet")
print(f"Loaded {len(matches)} matches")

player_names_cache = PROCESSED_DIR / "player_names_complete.parquet"

if player_names_cache.exists():
    # Load from cache if available
    print("Found cached player names, loading from disk")
    player_names_complete = pl.read_parquet(player_names_cache)
    print(f"Loaded {len(player_names_complete)} player names from cache")
    
else:
    # Extract names from all 306 matches
    print(f"Extracting player names from all {len(matches)} matches")
    print("Estimated time: 5-7 minutes")
    
    player_names_dict = {}
    failed_count = 0
    
    from tqdm.notebook import tqdm
    
    for match_id in tqdm(matches['matchId'], desc="Processing matches"):
        try:
            dataset = impect.load_open_data(match_id=match_id, competition_id=743)
            
            for team in dataset.metadata.teams:
                for player in team.players:
                    player_id_str = str(player.player_id)
                    
                    if player_id_str not in player_names_dict:
                        player_names_dict[player_id_str] = {
                            'player_name': player.name if hasattr(player, 'name') else None,
                            'jersey_no': player.jersey_no if hasattr(player, 'jersey_no') else None,
                        }
        except Exception as e:
            failed_count += 1
            continue
    
    # Convert to dataframe
    player_names_complete = pl.DataFrame([
        {
            'player_id': pid,
            'player_name': info['player_name'],
            'jersey_no': info['jersey_no']
        }
        for pid, info in player_names_dict.items()
    ])
    
    # Save to cache
    player_names_complete.write_parquet(player_names_cache)
    
    print(f"\nExtracted {len(player_names_complete)} player names")
    if failed_count > 0:
        print(f"Failed to load {failed_count} matches")
    print(f"Saved to cache: {player_names_cache}")

# Verify data quality
null_names = player_names_complete.filter(pl.col('player_name').is_null())

print(f"\nPlayer name extraction summary:")
print(f"  Total players: {len(player_names_complete)}")
print(f"  With names: {len(player_names_complete) - len(null_names)}")
print(f"  Missing names: {len(null_names)}")

if len(null_names) > 0 and len(null_names) < 10:
    print(f"\nPlayers with missing names:")
    print(null_names.head())

print("\nPlayer names ready for analysis")

Loading complete player names from match metadata
Loaded 306 matches
Found cached player names, loading from disk
Loaded 570 player names from cache

Player name extraction summary:
  Total players: 570
  With names: 570
  Missing names: 0

Player names ready for analysis


### Position Classification

Classify players into forwards, midfielders, and defenders based on their season-average field position (x-coordinate), enabling position-specific metric analysis.

In [7]:
# Cell 7: Classify player positions based on average field position

print("Classifying player positions based on average field position")

# Calculate season-average position for each player
player_positions = (
    all_events
    .filter(pl.col('player_id').is_not_null())
    .group_by('player_id')
    .agg([
        pl.mean('coordinates_x').alias('season_avg_x'),
        pl.mean('coordinates_y').alias('season_avg_y'),
        pl.len().alias('total_events'),
    ])
)

print(f"Calculated average positions for {len(player_positions)} players")

# Position classification function
def classify_position(avg_x):
    """
    Classify player position based on average x-coordinate.
    
    Thresholds (secondspectrum coordinates):
    - Forward: x > 10 (primarily in opponent's half)
    - Midfielder: -10 ≤ x ≤ 10 (central areas)
    - Defender: x < -10 (primarily in own half)
    
    Args:
        avg_x: Average x-coordinate across all events
        
    Returns:
        str: 'forward', 'midfielder', or 'defender'
    """
    if avg_x is None:
        return 'midfielder'
    
    if avg_x > 10:
        return 'forward'
    elif avg_x < -10:
        return 'defender'
    else:
        return 'midfielder'

# Apply position classification to all players
positions = [classify_position(x) for x in player_positions['season_avg_x']]
player_positions = player_positions.with_columns([
    pl.Series('position', positions)
])

# Display position distribution
print("\nPosition distribution:")
position_dist = player_positions.group_by('position').agg([
    pl.len().alias('count'),
    pl.mean('season_avg_x').alias('avg_x'),
    pl.min('season_avg_x').alias('min_x'),
    pl.max('season_avg_x').alias('max_x'),
]).sort('position')

print(position_dist)

print("\nPosition classification complete")

Classifying player positions based on average field position
Calculated average positions for 493 players

Position distribution:
shape: (3, 5)
┌────────────┬───────┬────────────┬────────────┬────────────┐
│ position   ┆ count ┆ avg_x      ┆ min_x      ┆ max_x      │
│ ---        ┆ ---   ┆ ---        ┆ ---        ┆ ---        │
│ str        ┆ u32   ┆ f64        ┆ f64        ┆ f64        │
╞════════════╪═══════╪════════════╪════════════╪════════════╡
│ defender   ┆ 120   ┆ -22.882705 ┆ -43.794366 ┆ -10.444847 │
│ forward    ┆ 121   ┆ 14.079463  ┆ 10.031763  ┆ 41.833333  │
│ midfielder ┆ 252   ┆ 1.127405   ┆ -9.782953  ┆ 9.987798   │
└────────────┴───────┴────────────┴────────────┴────────────┘

Position classification complete


### Aggregate Metrics by Player

Sum metric points across all events for each player and join with player names and positions to create comprehensive player profiles.
```


In [8]:
# Cell 8: Aggregate metric points by player

print("Aggregating metric points by player")

metric_columns = [
    'finishing', 'chance_creation', 'ball_progression', 'dribbling',
    'ball_winning', 'defensive_actions', 'passing_accuracy', 'long_passing'
]

# Aggregate all metric points by player
player_aggregated = (
    all_events
    .filter(pl.col('player_id').is_not_null())
    .group_by('player_id')
    .agg([
        # Sum all 8 metric point totals
        pl.sum('finishing_points').alias('total_finishing'),
        pl.sum('chance_creation_points').alias('total_chance_creation'),
        pl.sum('ball_progression_points').alias('total_ball_progression'),
        pl.sum('dribbling_points').alias('total_dribbling'),
        pl.sum('ball_winning_points').alias('total_ball_winning'),
        pl.sum('defensive_actions_points').alias('total_defensive_actions'),
        pl.sum('passing_accuracy_points').alias('total_passing_accuracy'),
        pl.sum('long_passing_points').alias('total_long_passing'),
        
        # Event and match counts
        pl.len().alias('total_events'),
        pl.col('match_id').n_unique().alias('matches_played'),
    ])
)

print(f"Aggregated metrics for {len(player_aggregated)} players")

# Join player names
player_aggregated = player_aggregated.join(
    player_names_complete.select(['player_id', 'player_name']),
    on='player_id',
    how='left'
)

# Join positions
player_aggregated = player_aggregated.join(
    player_positions.select(['player_id', 'position', 'season_avg_x']),
    on='player_id',
    how='left'
)

# Verify data integrity
print(f"\nData integrity check:")
print(f"  Total records: {len(player_aggregated)}")
print(f"  Unique players: {player_aggregated['player_id'].n_unique()}")

if len(player_aggregated) == player_aggregated['player_id'].n_unique():
    print(f"  No duplicates detected")
else:
    print(f"  Warning: Duplicate player_ids found")

# Display sample
print("\nSample of aggregated data:")
print(player_aggregated.select([
    'player_name', 'position', 'matches_played', 'total_events',
    'total_finishing', 'total_ball_progression', 'total_ball_winning'
]).head(5))

print("\nPlayer aggregation complete")

Aggregating metric points by player
Aggregated metrics for 493 players

Data integrity check:
  Total records: 493
  Unique players: 493
  No duplicates detected

Sample of aggregated data:
shape: (5, 7)
┌──────────────┬──────────┬──────────────┬──────────────┬──────────────┬─────────────┬─────────────┐
│ player_name  ┆ position ┆ matches_play ┆ total_events ┆ total_finish ┆ total_ball_ ┆ total_ball_ │
│ ---          ┆ ---      ┆ ed           ┆ ---          ┆ ing          ┆ progression ┆ winning     │
│ str          ┆ str      ┆ ---          ┆ u32          ┆ ---          ┆ ---         ┆ ---         │
│              ┆          ┆ u32          ┆              ┆ f64          ┆ f64         ┆ f64         │
╞══════════════╪══════════╪══════════════╪══════════════╪══════════════╪═════════════╪═════════════╡
│ Paxten       ┆ forward  ┆ 7            ┆ 181          ┆ 4.0          ┆ 20.5        ┆ 23.0        │
│ Aaronson     ┆          ┆              ┆              ┆              ┆             ┆   

In [9]:
# Cell 9: Estimate playing time and calculate per-90 metrics

print("Calculating per-90 metrics")

# Estimate minutes based on event participation
avg_events_per_match = all_events.group_by('match_id').agg(
    pl.len().alias('events')
)['events'].mean()

print(f"Average events per match: {avg_events_per_match:.0f}")

# Use baseline: starter with ~35 events plays ~90 minutes
avg_starter_events_per_match = 35

# Calculate events per match for each player
player_aggregated = player_aggregated.with_columns([
    (pl.col('total_events') / pl.col('matches_played')).alias('events_per_match'),
])

# Estimate minutes per match (capped at 90)
player_aggregated = player_aggregated.with_columns([
    (
        (pl.col('events_per_match') / avg_starter_events_per_match) * 90
    ).clip(0, 90).alias('estimated_minutes_per_match'),
])

# Calculate total estimated minutes
player_aggregated = player_aggregated.with_columns([
    (pl.col('estimated_minutes_per_match') * pl.col('matches_played')).alias('estimated_total_minutes')
])

print("Estimated playing time for all players")

# Display minutes distribution
print(f"\nMinutes distribution:")
print(f"  Minimum: {player_aggregated['estimated_total_minutes'].min():.0f} minutes")
print(f"  Maximum: {player_aggregated['estimated_total_minutes'].max():.0f} minutes")
print(f"  Mean: {player_aggregated['estimated_total_minutes'].mean():.0f} minutes")
print(f"  Median: {player_aggregated['estimated_total_minutes'].median():.0f} minutes")

# Filter for players with substantial minutes
MIN_MINUTES = 500

player_aggregated = player_aggregated.filter(
    pl.col('estimated_total_minutes') >= MIN_MINUTES
)

print(f"\nFiltered to {len(player_aggregated)} players with >{MIN_MINUTES} minutes")

# Calculate per-90 for each metric
metric_columns = [
    'finishing', 'chance_creation', 'ball_progression', 'dribbling',
    'ball_winning', 'defensive_actions', 'passing_accuracy', 'long_passing'
]

for metric in metric_columns:
    player_aggregated = player_aggregated.with_columns([
        ((pl.col(f'total_{metric}') / pl.col('estimated_total_minutes')) * 90)
        .alias(f'{metric}_per90')
    ])

print("Calculated per-90 rates for all 8 metrics")

# Display per-90 metric ranges
print("\nPer-90 metric distributions:")
for metric in metric_columns:
    col = f'{metric}_per90'
    mean_val = player_aggregated[col].mean()
    max_val = player_aggregated[col].max()
    min_val = player_aggregated[col].min()
    print(f"  {metric:25s}: Range=[{min_val:>5.2f}, {max_val:>5.2f}], Mean={mean_val:>5.2f}")

# Show top performers in key metrics
print("\nTop 5 by finishing per-90:")
print(player_aggregated.select([
    'player_name', 'position', 'finishing_per90', 'matches_played', 'estimated_total_minutes'
]).sort('finishing_per90', descending=True).head(5))

print("\nTop 5 by ball progression per-90:")
print(player_aggregated.select([
    'player_name', 'position', 'ball_progression_per90', 'matches_played', 'estimated_total_minutes'
]).sort('ball_progression_per90', descending=True).head(5))

print("\nPer-90 normalization complete")

Calculating per-90 metrics
Average events per match: 3147
Estimated playing time for all players

Minutes distribution:
  Minimum: 8 minutes
  Maximum: 3060 minutes
  Mean: 1694 minutes
  Median: 1890 minutes

Filtered to 395 players with >500 minutes
Calculated per-90 rates for all 8 metrics

Per-90 metric distributions:
  finishing                : Range=[ 0.00, 13.83], Mean= 1.46
  chance_creation          : Range=[ 0.00, 22.16], Mean= 3.47
  ball_progression         : Range=[ 1.45, 57.26], Mean=17.12
  dribbling                : Range=[ 0.40, 10.57], Mean= 3.73
  ball_winning             : Range=[ 0.00, 18.77], Mean= 7.11
  defensive_actions        : Range=[ 0.40, 21.44], Mean= 6.91
  passing_accuracy         : Range=[ 2.53, 77.04], Mean=16.83
  long_passing             : Range=[-0.48, 14.52], Mean= 3.03

Top 5 by finishing per-90:
shape: (5, 5)
┌─────────────────┬──────────┬─────────────────┬────────────────┬─────────────────────────┐
│ player_name     ┆ position ┆ finishing_per90

### Within-Position Percentile Ratings

Calculate percentile-based ratings (0-99 scale) for each metric within position groups, ensuring fair comparison by evaluating players against peers in the same tactical role.

In [10]:
# Cell 10: Calculate within-position percentile ratings

print("Calculating within-position percentile ratings (0-99 scale)")

from scipy.stats import percentileofscore

# Define metric columns
metric_columns = [
    'finishing', 'chance_creation', 'ball_progression', 'dribbling',
    'ball_winning', 'defensive_actions', 'passing_accuracy', 'long_passing'
]

print("\nCalculating percentiles within positions...")

# For each metric, calculate percentile within each position
for metric in metric_columns:
    per90_col = f'{metric}_per90'
    rating_col = f'{metric}_rating'
    
    all_ratings = []
    
    # Process each position separately
    for position in ['forward', 'midfielder', 'defender']:
        position_players = player_aggregated.filter(pl.col('position') == position)
        
        if len(position_players) == 0:
            continue
        
        # Get per-90 values for this position
        per90_values = position_players[per90_col].to_numpy()
        
        # Calculate percentile for each player within their position
        percentiles = []
        for value in per90_values:
            pct = percentileofscore(per90_values, value, kind='rank')
            percentiles.append(pct)
        
        # Scale to 0-99 range
        percentiles_scaled = [p * 0.99 for p in percentiles]
        
        # Store with player_ids
        for i, player_id in enumerate(position_players['player_id']):
            all_ratings.append({
                'player_id': player_id,
                rating_col: percentiles_scaled[i]
            })
    
    # Join ratings back to main dataframe
    ratings_df = pl.DataFrame(all_ratings)
    player_aggregated = player_aggregated.join(ratings_df, on='player_id', how='left')

print("\nAll 8 metrics scaled to 0-99 within each position")

# Verify rating ranges
print("\nRating distributions:")
for metric in metric_columns:
    rating_col = f'{metric}_rating'
    min_r = player_aggregated[rating_col].min()
    max_r = player_aggregated[rating_col].max()
    mean_r = player_aggregated[rating_col].mean()
    
    print(f"  {metric:25s}: [{min_r:>5.1f}, {max_r:>5.1f}], Mean={mean_r:>5.1f}")

# Display top performers in key metrics
print("\nTop 5 performers by metric (within-position percentiles):")

for metric in ['finishing', 'chance_creation', 'ball_winning', 'ball_progression']:
    rating_col = f'{metric}_rating'
    per90_col = f'{metric}_per90'
    
    print(f"\n{metric.replace('_', ' ').title()}:")
    print(player_aggregated.select([
        'player_name', 'position', rating_col, per90_col, 'matches_played'
    ]).sort(rating_col, descending=True).head(5))

print("\nWithin-position rating calculation complete")

Calculating within-position percentile ratings (0-99 scale)

Calculating percentiles within positions...

All 8 metrics scaled to 0-99 within each position

Rating distributions:
  finishing                : [  0.7,  99.0], Mean= 49.9
  chance_creation          : [  0.7,  99.0], Mean= 49.9
  ball_progression         : [  0.5,  99.0], Mean= 49.9
  dribbling                : [  0.5,  99.0], Mean= 49.9
  ball_winning             : [  0.5,  99.0], Mean= 49.9
  defensive_actions        : [  0.5,  99.0], Mean= 49.9
  passing_accuracy         : [  0.5,  99.0], Mean= 49.9
  long_passing             : [  0.5,  99.0], Mean= 49.9

Top 5 performers by metric (within-position percentiles):

Finishing:
shape: (5, 5)
┌─────────────────────┬────────────┬──────────────────┬─────────────────┬────────────────┐
│ player_name         ┆ position   ┆ finishing_rating ┆ finishing_per90 ┆ matches_played │
│ ---                 ┆ ---        ┆ ---              ┆ ---             ┆ ---            │
│ str          

### Within-Position Percentile Ratings

Calculate percentile-based ratings (0-99 scale) for each metric within position groups, with null values for metrics not relevant to specific positions (e.g., finishing for defenders).

In [11]:
# Cell 10: Calculate within-position percentile ratings

print("Calculating within-position percentile ratings (0-99 scale)")

from scipy.stats import percentileofscore

# Define which metrics are relevant for each position
POSITION_RELEVANT_METRICS = {
    'forward': ['finishing', 'chance_creation', 'dribbling', 'ball_progression'],
    'midfielder': ['finishing', 'chance_creation', 'ball_progression', 'dribbling',
                   'ball_winning', 'defensive_actions', 'passing_accuracy'],
    'defender': ['ball_winning', 'defensive_actions', 'ball_progression',
                 'passing_accuracy', 'long_passing']
}

print("\nPosition-relevant metrics:")
for pos, metrics in POSITION_RELEVANT_METRICS.items():
    print(f"  {pos:10s} ({len(metrics)} metrics): {', '.join(metrics)}")

all_metrics = [
    'finishing', 'chance_creation', 'ball_progression', 'dribbling',
    'ball_winning', 'defensive_actions', 'passing_accuracy', 'long_passing'
]

# Calculate percentile ratings for each metric
print("\nCalculating percentiles within positions...")

for metric in all_metrics:
    per90_col = f'{metric}_per90'
    rating_col = f'{metric}_rating'
    
    # Initialize with NULL
    player_aggregated = player_aggregated.with_columns([
        pl.lit(None, dtype=pl.Float64).alias(rating_col)
    ])
    
    # Process each position separately
    for position in ['forward', 'midfielder', 'defender']:
        # Skip if metric not relevant for this position
        if metric not in POSITION_RELEVANT_METRICS[position]:
            continue
        
        # Get players in this position
        position_mask = player_aggregated['position'] == position
        position_indices = [i for i, val in enumerate(position_mask) if val]
        
        if len(position_indices) == 0:
            continue
        
        # Get per-90 values
        position_players = player_aggregated.filter(pl.col('position') == position)
        per90_values = position_players[per90_col].to_numpy()
        
        # Calculate percentiles
        percentiles = []
        for value in per90_values:
            pct = percentileofscore(per90_values, value, kind='rank')
            scaled = round(pct * 0.99, 1)  # Scale to 0-99
            percentiles.append(scaled)
        
        # Update ratings
        current_ratings = player_aggregated[rating_col].to_list()
        for i, pos_idx in enumerate(position_indices):
            current_ratings[pos_idx] = percentiles[i]
        
        player_aggregated = player_aggregated.with_columns([
            pl.Series(rating_col, current_ratings)
        ])
    
    print(f"  Calculated {metric}")

print("\nAll percentile ratings calculated")

# Verify ratings are properly bounded
print("\nRating range verification:")
for metric in all_metrics:
    rating_col = f'{metric}_rating'
    non_null = player_aggregated[rating_col].drop_nulls()
    
    if len(non_null) > 0:
        max_r = non_null.max()
        mean_r = non_null.mean()
        print(f"  {metric:25s}: Max={max_r:>5.1f}, Mean={mean_r:>5.1f}")

# Display top performers in key metrics
print("\nTop 5 performers by key metrics:")

for metric in ['finishing', 'chance_creation', 'ball_winning', 'ball_progression']:
    rating_col = f'{metric}_rating'
    per90_col = f'{metric}_per90'
    
    print(f"\n{metric.replace('_', ' ').title()}:")
    print(player_aggregated
          .filter(pl.col(rating_col).is_not_null())
          .select(['player_name', 'position', rating_col, per90_col, 'matches_played'])
          .sort(rating_col, descending=True)
          .head(5))

print("\nWithin-position rating calculation complete")

Calculating within-position percentile ratings (0-99 scale)

Position-relevant metrics:
  forward    (4 metrics): finishing, chance_creation, dribbling, ball_progression
  midfielder (7 metrics): finishing, chance_creation, ball_progression, dribbling, ball_winning, defensive_actions, passing_accuracy
  defender   (5 metrics): ball_winning, defensive_actions, ball_progression, passing_accuracy, long_passing

Calculating percentiles within positions...
  Calculated finishing
  Calculated chance_creation
  Calculated ball_progression
  Calculated dribbling
  Calculated ball_winning
  Calculated defensive_actions
  Calculated passing_accuracy
  Calculated long_passing

All percentile ratings calculated

Rating range verification:
  finishing                : Max= 99.0, Mean= 49.8
  chance_creation          : Max= 99.0, Mean= 49.8
  ball_progression         : Max= 99.0, Mean= 49.9
  dribbling                : Max= 99.0, Mean= 49.8
  ball_winning             : Max= 99.0, Mean= 49.8
  defens

### Calculate Overall Ratings

Combine individual metrics into overall ratings using position-specific weights, creating both cross-position ratings (for market value analysis) and within-position percentiles (for role-specific comparison).

In [12]:
# Cell 11: Calculate overall ratings (cross-position and within-position)

print("Calculating overall ratings")

# Define position-specific weights for overall rating
# Weights sum to 1.0, using only relevant metrics per position
POSITION_OVERALL_WEIGHTS = {
    'forward': {
        'finishing': 0.45,
        'chance_creation': 0.30,
        'dribbling': 0.20,
        'ball_progression': 0.05,
    },
    'midfielder': {
        'ball_progression': 0.30,
        'chance_creation': 0.20,
        'passing_accuracy': 0.15,
        'ball_winning': 0.15,
        'dribbling': 0.10,
        'defensive_actions': 0.05,
        'finishing': 0.05,
    },
    'defender': {
        'ball_winning': 0.35,
        'defensive_actions': 0.30,
        'ball_progression': 0.20,
        'passing_accuracy': 0.10,
        'long_passing': 0.05,
    }
}

print("\nPosition-specific weights for overall rating:")
for position, weights in POSITION_OVERALL_WEIGHTS.items():
    print(f"\n{position.title()}:")
    for metric, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
        print(f"  {metric:25s}: {weight:>5.1%}")

# Calculate cross-position overall rating
print("\nCalculating cross-position overall ratings...")

def calculate_cross_position_overall(row):
    """
    Calculate overall rating using position-specific weights.
    Allows comparison across all positions.
    """
    position = row['position']
    
    if position not in POSITION_OVERALL_WEIGHTS:
        return None
    
    weights = POSITION_OVERALL_WEIGHTS[position]
    overall = 0.0
    
    for metric, weight in weights.items():
        rating = row.get(f'{metric}_rating')
        if rating is not None:
            overall += rating * weight
    
    return round(overall, 1)

# Apply to all players
cross_position_overall = []
for row in player_aggregated.iter_rows(named=True):
    cross_position_overall.append(calculate_cross_position_overall(row))

player_aggregated = player_aggregated.with_columns([
    pl.Series('overall_rating', cross_position_overall)
])

print("Cross-position overall ratings calculated")

# Display distribution
print(f"\nCross-position overall rating distribution:")
print(f"  Min: {player_aggregated['overall_rating'].min():.1f}")
print(f"  Max: {player_aggregated['overall_rating'].max():.1f}")
print(f"  Mean: {player_aggregated['overall_rating'].mean():.1f}")
print(f"  Median: {player_aggregated['overall_rating'].median():.1f}")

# Calculate within-position overall percentiles
print("\nCalculating within-position overall percentiles...")

within_position_overall = []

for position in ['forward', 'midfielder', 'defender']:
    position_players = player_aggregated.filter(pl.col('position') == position)
    
    if len(position_players) == 0:
        continue
    
    overall_values = position_players['overall_rating'].to_numpy()
    
    for i, row in enumerate(position_players.iter_rows(named=True)):
        overall_val = overall_values[i]
        pct = percentileofscore(overall_values, overall_val, kind='rank')
        scaled = round(pct * 0.99, 1)
        
        within_position_overall.append({
            'player_id': row['player_id'],
            'within_position_overall': scaled
        })

# Join back
within_df = pl.DataFrame(within_position_overall)
player_aggregated = player_aggregated.join(within_df, on='player_id', how='left')

print("Within-position overall percentiles calculated")

# Display top 20 cross-position
print("\nTop 20 players (cross-position overall rating):")
print("Used for market value comparison - all positions on same scale\n")

top_20_cross = player_aggregated.select([
    'player_name', 'position', 'overall_rating', 'within_position_overall',
    'finishing_rating', 'chance_creation_rating', 'ball_progression_rating',
    'ball_winning_rating', 'matches_played'
]).sort('overall_rating', descending=True).head(20)

print(top_20_cross)

# Position distribution in top 20
print("\nPosition distribution in top 20:")
pos_dist = top_20_cross.group_by('position').agg(pl.len().alias('count')).sort('position')
print(pos_dist)

# Top 10 per position (within-position ranking)
print("\nTop 10 by position (within-position overall):")
print("Best in their role - 99 = best forward/midfielder/defender\n")

for position in ['forward', 'midfielder', 'defender']:
    print(f"{position.title()}s:")
    print(player_aggregated
          .filter(pl.col('position') == position)
          .select(['player_name', 'within_position_overall', 'overall_rating', 'matches_played'])
          .sort('within_position_overall', descending=True)
          .head(10))
    print()

print("Overall rating calculation complete")

Calculating overall ratings

Position-specific weights for overall rating:

Forward:
  finishing                : 45.0%
  chance_creation          : 30.0%
  dribbling                : 20.0%
  ball_progression         :  5.0%

Midfielder:
  ball_progression         : 30.0%
  chance_creation          : 20.0%
  passing_accuracy         : 15.0%
  ball_winning             : 15.0%
  dribbling                : 10.0%
  defensive_actions        :  5.0%
  finishing                :  5.0%

Defender:
  ball_winning             : 35.0%
  defensive_actions        : 30.0%
  ball_progression         : 20.0%
  passing_accuracy         : 10.0%
  long_passing             :  5.0%

Calculating cross-position overall ratings...
Cross-position overall ratings calculated

Cross-position overall rating distribution:
  Min: 2.0
  Max: 93.0
  Mean: 49.9
  Median: 49.0

Calculating within-position overall percentiles...
Within-position overall percentiles calculated

Top 20 players (cross-position overall rating)

### Generate Market Value Lookup List

Export top 300 players for manual market value collection from external sources (Transfermarkt), enabling validation of ratings against real-world player valuations.

In [13]:
# Cell 12A: Generate player list for market value lookup

print("Generating player list for market value lookup")

# Select top 300 players by overall rating for market value analysis
top_300_lookup = player_aggregated.sort('overall_rating', descending=True).head(300)

# Save as plain text file (one name per line for easy lookup)
lookup_file = DATA_DIR / "top_300_for_market_lookup.txt"

with open(lookup_file, 'w', encoding='utf-8') as f:
    for name in top_300_lookup['player_name']:
        f.write(f"{name}\n")

print(f"Saved {len(top_300_lookup)} player names to: {lookup_file}")

# Save detailed version with ratings for reference
lookup_with_info = DATA_DIR / "top_300_with_ratings.csv"
top_300_lookup.select([
    'player_name', 'position', 'overall_rating', 'matches_played'
]).write_csv(lookup_with_info)

print(f"Saved detailed version to: {lookup_with_info}")

print("\nPlayer list ready for market value collection")

Generating player list for market value lookup
Saved 300 player names to: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/top_300_for_market_lookup.txt
Saved detailed version to: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/top_300_with_ratings.csv

Player list ready for market value collection


### Market Value Analysis

Load player market values and calculate value scores to identify undervalued (high rating, low price) and overvalued (low rating, high price) players.

In [14]:
# Cell 12: Load market values and calculate value scores

print("Loading market values and calculating value scores")

# Use DATA_DIR from Cell 2 (handles Notebooks/ subfolder correctly)
market_values_file = DATA_DIR / "market_values.csv"

print(f"Looking for: {market_values_file}")
print(f"File exists: {market_values_file.exists()}")

if market_values_file.exists():
    print(f"Loading market values from: {market_values_file}")
    
    # Load and clean market values
    market_values = pl.read_csv(market_values_file)
    print(f"Total entries: {len(market_values)}")
    
    # Ensure numeric type
    market_values = market_values.with_columns([
        pl.col('market_value_millions').cast(pl.Float64, strict=False)
    ])
    
    market_values_clean = market_values.filter(
        pl.col('market_value_millions').is_not_null()
    )
    
    print(f"Clean numeric values: {len(market_values_clean)}")
    
    # Sample preview
    print("\nSample of market values:")
    print(market_values_clean.head(5))
    
    # Join with player ratings
    player_final = player_aggregated.join(
        market_values_clean,
        on='player_name',
        how='left'
    )
    
    matched = player_final.filter(pl.col('market_value_millions').is_not_null())
    coverage = len(matched) / len(player_aggregated) * 100
    print(f"\nMatched {len(matched)}/{len(player_aggregated)} players ({coverage:.1f}% coverage)")
    
    # Calculate value metrics
    player_final = player_final.with_columns([
        # Value score: rating per million euros
        (pl.col('overall_rating') / pl.col('market_value_millions')).alias('value_score'),
        
        # Expected rating based on market value (€15M = 50 rating baseline)
        ((pl.col('market_value_millions') / 15.0) * 50).clip(0, 95).alias('expected_rating'),
        
        # Rating vs expected
        (pl.col('overall_rating') - ((pl.col('market_value_millions') / 15.0) * 50).clip(0, 95)).alias('rating_vs_expected'),
    ])
    
    print("Value metrics calculated")
    
    # Market value statistics
    print(f"\nMarket value distribution:")
    print(f"  Range: €{matched['market_value_millions'].min():.1f}M - €{matched['market_value_millions'].max():.1f}M")
    print(f"  Mean: €{matched['market_value_millions'].mean():.1f}M")
    print(f"  Median: €{matched['market_value_millions'].median():.1f}M")
    
    # Value score statistics
    print(f"\nValue score distribution:")
    valid_scores = player_final.filter(pl.col('value_score').is_not_null())['value_score']
    print(f"  Range: {valid_scores.min():.2f} - {valid_scores.max():.2f} rating/million")
    print(f"  Mean: {valid_scores.mean():.2f} rating/million")
    
    # Best value players (undervalued)
    print("\nTop 10 best value players (undervalued):")
    
    best_value = player_final.filter(
        pl.col('value_score').is_not_null()
    ).select([
        'player_name', 'position', 'overall_rating', 
        'market_value_millions', 'value_score', 
        'rating_vs_expected', 'matches_played'
    ]).sort('value_score', descending=True).head(10)
    
    print(best_value)
    
    # Worst value players (overvalued)
    print("\nTop 10 worst value players (overvalued, min €15M):")
    
    worst_value = player_final.filter(
        (pl.col('value_score').is_not_null()) &
        (pl.col('market_value_millions') >= 15)
    ).select([
        'player_name', 'position', 'overall_rating', 
        'market_value_millions', 'value_score',
        'rating_vs_expected', 'matches_played'
    ]).sort('value_score', descending=False).head(10)
    
    print(worst_value)
    
else:
    print(f"Market values file not found: {market_values_file}")
    print(f"Absolute path: {market_values_file.absolute()}")
    print("Skipping market value analysis")
    player_final = player_aggregated

print("\nMarket value analysis complete")

Loading market values and calculating value scores
Looking for: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/market_values.csv
File exists: True
Loading market values from: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/market_values.csv
Total entries: 300
Clean numeric values: 300

Sample of market values:
shape: (5, 2)
┌────────────────────┬───────────────────────┐
│ player_name        ┆ market_value_millions │
│ ---                ┆ ---                   │
│ str                ┆ f64                   │
╞════════════════════╪═══════════════════════╡
│ Andrej Kramaric    ┆ 6.0                   │
│ Timo Hübers        ┆ 5.5                   │
│ Min-jae Kim        ┆ 45.0                  │
│ Florian Wirtz      ┆ 130.0                 │
│ Nico Schlotterbeck ┆ 40.0                  │
└────────────────────┴───────────────────────┘

Matched 300/395 players (75.9% coverage)
Value metrics calculated

Market value distribution:
  Range: €0.2M - €130.0M
  Mean: €13.0M
  Med

### Value Categorization

Categorize players into value propositions (hidden gems, great value, overpriced, etc.) based on the relationship between performance ratings and market valuations.

In [15]:
# Cell 13: Categorize players by value proposition

print("Categorizing players by value proposition")

# Create value categories based on rating and market value
player_final = player_final.with_columns([
    pl.when(
        (pl.col('overall_rating') >= 85) & (pl.col('market_value_millions') < 15)
    ).then(pl.lit("Hidden Gem (Elite Performance, Low Price)"))
    
    .when(
        (pl.col('overall_rating') >= 75) & (pl.col('market_value_millions') < 20)
    ).then(pl.lit("Great Value (High Performance, Reasonable Price)"))
    
    .when(
        (pl.col('overall_rating') >= 85) & (pl.col('market_value_millions') >= 50)
    ).then(pl.lit("Elite (Expensive but Worth It)"))
    
    .when(
        (pl.col('overall_rating') < 70) & (pl.col('market_value_millions') >= 30)
    ).then(pl.lit("Overpriced (Expensive, Low Performance)"))
    
    .when(
        (pl.col('overall_rating') >= 70) & (pl.col('market_value_millions') >= 25)
    ).then(pl.lit("Fair Value (Good Performance, Fair Price)"))
    
    .otherwise(pl.lit("Standard"))
    
    .alias('value_category')
])

# Display category distribution
print("\nValue category distribution:")
categories = player_final.filter(
    pl.col('market_value_millions').is_not_null()
).group_by('value_category').agg([
    pl.len().alias('count'),
    pl.mean('overall_rating').alias('avg_rating'),
    pl.mean('market_value_millions').alias('avg_value')
]).sort('count', descending=True)

print(categories)

# Hidden gems: Elite performers at budget prices
print("\nHidden Gems: Elite performers at budget prices")
print("(Rating ≥75, Market Value <€15M)")

hidden_gems = player_final.filter(
    (pl.col('overall_rating') >= 75) &
    (pl.col('market_value_millions') < 15) &
    (pl.col('market_value_millions').is_not_null())
).select([
    'player_name', 'position', 'overall_rating', 
    'market_value_millions', 'value_score',
    'rating_vs_expected', 'matches_played'
]).sort('overall_rating', descending=True)

print(hidden_gems)
print(f"\nFound {len(hidden_gems)} hidden gems")

# Great value: Good performers at affordable prices
print("\nGreat value players: Good performers at affordable prices")
print("(Rating 70-85, Market Value <€20M)")

great_value = player_final.filter(
    (pl.col('overall_rating') >= 70) &
    (pl.col('overall_rating') < 85) &
    (pl.col('market_value_millions') < 20) &
    (pl.col('market_value_millions').is_not_null())
).select([
    'player_name', 'position', 'overall_rating', 
    'market_value_millions', 'value_score', 'matches_played'
]).sort('overall_rating', descending=True).head(15)

print(great_value)

# Overpriced: Expensive but underperforming
print("\nOverpriced players: Expensive but underperforming")
print("(Rating <75, Market Value ≥€30M)")

overpriced = player_final.filter(
    (pl.col('overall_rating') < 75) &
    (pl.col('market_value_millions') >= 30) &
    (pl.col('market_value_millions').is_not_null())
).select([
    'player_name', 'position', 'overall_rating', 
    'market_value_millions', 'value_score',
    'rating_vs_expected', 'matches_played'
]).sort('market_value_millions', descending=True)

if len(overpriced) > 0:
    print(overpriced)
else:
    print("No players in this category")

# Top 10 overall value scores
print("\nTop 10 best value overall (minimum rating 70):")
print("Best performance-to-price ratio for quality players")

elite_value = player_final.filter(
    (pl.col('overall_rating') >= 70) &
    (pl.col('value_score').is_not_null())
).select([
    'player_name', 'position', 'overall_rating', 
    'market_value_millions', 'value_score',
    'rating_vs_expected', 'matches_played'
]).sort('value_score', descending=True).head(10)

print(elite_value)

print("\nValue categorization complete")

Categorizing players by value proposition

Value category distribution:
shape: (6, 4)
┌─────────────────────────────────┬───────┬────────────┬───────────┐
│ value_category                  ┆ count ┆ avg_rating ┆ avg_value │
│ ---                             ┆ ---   ┆ ---        ┆ ---       │
│ str                             ┆ u32   ┆ f64        ┆ f64       │
╞═════════════════════════════════╪═══════╪════════════╪═══════════╡
│ Standard                        ┆ 224   ┆ 52.789286  ┆ 6.94375   │
│ Fair Value (Good Performance, … ┆ 33    ┆ 79.836364  ┆ 41.212121 │
│ Great Value (High Performance,… ┆ 23    ┆ 78.93913   ┆ 9.108696  │
│ Overpriced (Expensive, Low Per… ┆ 10    ┆ 60.02      ┆ 40.5      │
│ Hidden Gem (Elite Performance,… ┆ 6     ┆ 89.433333  ┆ 7.0       │
│ Elite (Expensive but Worth It)  ┆ 4     ┆ 88.875     ┆ 82.5      │
└─────────────────────────────────┴───────┴────────────┴───────────┘

Hidden Gems: Elite performers at budget prices
(Rating ≥75, Market Value <€15M)
shape

### Save Final Player Ratings

Export complete player ratings dataset with all 8 granular metrics and overall ratings to disk for validation analysis and visualization.

In [16]:
# Cell 14: Save final player ratings dataset

print("Saving final player ratings dataset")

# Save complete dataset with all 8 metrics
player_final = player_aggregated  # Use the aggregated data (no market values)

player_final.write_parquet(PROCESSED_DIR / "player_ratings_with_8_metrics.parquet")
player_final.write_csv(PROCESSED_DIR / "player_ratings_with_8_metrics.csv")

print(f"Saved complete dataset:")
print(f"  {PROCESSED_DIR / 'player_ratings_with_8_metrics.parquet'}")
print(f"  {PROCESSED_DIR / 'player_ratings_with_8_metrics.csv'}")

# Summary statistics
print(f"\nFinal dataset summary:")
print(f"  Total players: {len(player_final)}")
print(f"  Forwards: {(player_final['position'] == 'forward').sum()}")
print(f"  Midfielders: {(player_final['position'] == 'midfielder').sum()}")
print(f"  Defenders: {(player_final['position'] == 'defender').sum()}")

print(f"\nMetrics included:")
print(f"  8 granular metrics: finishing, chance_creation, ball_progression, dribbling,")
print(f"                      ball_winning, defensive_actions, passing_accuracy, long_passing")
print(f"  2 overall ratings: overall_rating (cross-position), within_position_overall")
print(f"  Position-specific: Only relevant metrics scored per position")

print("\nPlayer ratings dataset saved")

Saving final player ratings dataset
Saved complete dataset:
  /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed/player_ratings_with_8_metrics.parquet
  /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed/player_ratings_with_8_metrics.csv

Final dataset summary:
  Total players: 395
  Forwards: 87
  Midfielders: 207
  Defenders: 101

Metrics included:
  8 granular metrics: finishing, chance_creation, ball_progression, dribbling,
                      ball_winning, defensive_actions, passing_accuracy, long_passing
  2 overall ratings: overall_rating (cross-position), within_position_overall
  Position-specific: Only relevant metrics scored per position

Player ratings dataset saved


In [17]:
# Cell 15: Extract ground truth statistics from events

print("Extracting ground truth statistics from event data")

# Calculate actual goals scored by each player
goals_data = (
    all_events
    .filter(
        (pl.col('event_type') == 'SHOT') & 
        (pl.col('result') == 'GOAL') &
        (pl.col('player_id').is_not_null())
    )
    .group_by('player_id')
    .agg([
        pl.len().alias('goals_actual')
    ])
)

print(f"Extracted goals for {len(goals_data)} players")

# Calculate actual assists (SHOT_ASSIST passes)
assists_data = (
    all_events
    .filter(
        (pl.col('event_type') == 'PASS') & 
        (pl.col('pass_type') == 'SHOT_ASSIST') &
        (pl.col('player_id').is_not_null())
    )
    .group_by('player_id')
    .agg([
        pl.len().alias('assists_actual')
    ])
)

print(f"Extracted assists for {len(assists_data)} players")

# Join with player ratings
player_final = player_aggregated.join(goals_data, on='player_id', how='left')
player_final = player_final.join(assists_data, on='player_id', how='left')

# Fill nulls with 0 (players with no goals/assists)
player_final = player_final.with_columns([
    pl.col('goals_actual').fill_null(0),
    pl.col('assists_actual').fill_null(0)
])

# Calculate per-90 rates for ground truth
player_final = player_final.with_columns([
    ((pl.col('goals_actual') / pl.col('estimated_total_minutes')) * 90).alias('goals_per90_actual'),
    ((pl.col('assists_actual') / pl.col('estimated_total_minutes')) * 90).alias('assists_per90_actual')
])

# Display statistics
print("\nGround truth statistics:")
print(f"  Players with goals: {(player_final['goals_actual'] > 0).sum()}")
print(f"  Players with assists: {(player_final['assists_actual'] > 0).sum()}")
print(f"  Total goals in dataset: {player_final['goals_actual'].sum()}")
print(f"  Total assists in dataset: {player_final['assists_actual'].sum()}")

# Show top scorers and assisters
print("\nTop 5 goal scorers:")
print(player_final.select([
    'player_name', 'position', 'goals_actual', 'goals_per90_actual', 'finishing_rating'
]).sort('goals_actual', descending=True).head(5))

print("\nTop 5 assist providers:")
print(player_final.select([
    'player_name', 'position', 'assists_actual', 'assists_per90_actual', 'chance_creation_rating'
]).sort('assists_actual', descending=True).head(5))

print("\nGround truth extraction complete")

Extracting ground truth statistics from event data
Extracted goals for 259 players
Extracted assists for 360 players

Ground truth statistics:
  Players with goals: 251
  Players with assists: 345
  Total goals in dataset: 951
  Total assists in dataset: 2828

Top 5 goal scorers:
shape: (5, 5)
┌──────────────────┬──────────┬──────────────┬────────────────────┬──────────────────┐
│ player_name      ┆ position ┆ goals_actual ┆ goals_per90_actual ┆ finishing_rating │
│ ---              ┆ ---      ┆ ---          ┆ ---                ┆ ---              │
│ str              ┆ str      ┆ u32          ┆ f64                ┆ f64              │
╞══════════════════╪══════════╪══════════════╪════════════════════╪══════════════════╡
│ Harry Kane       ┆ forward  ┆ 36           ┆ 1.125              ┆ 99.0             │
│ Serhou Guirassy  ┆ forward  ┆ 28           ┆ 1.0                ┆ 97.9             │
│ Loïs Openda      ┆ forward  ┆ 24           ┆ 0.705882           ┆ 96.7             │
│ Deniz U

In [18]:
# Cell 16: Validate metrics against ground truth

print("Validating metrics against ground truth statistics")

from scipy.stats import pearsonr
import numpy as np

# Validation 1: Finishing rating vs actual goals (forwards)
print("\nValidation 1: Finishing Rating vs Actual Goals")
print("Scope: Forwards only")

forwards_with_finishing = player_final.filter(
    (pl.col('position') == 'forward') &
    (pl.col('finishing_rating').is_not_null())
)

finishing_ratings = forwards_with_finishing['finishing_rating'].to_numpy()
goals_per90 = forwards_with_finishing['goals_per90_actual'].to_numpy()

# Remove NaN values
mask = ~(np.isnan(finishing_ratings) | np.isnan(goals_per90))
finishing_clean = finishing_ratings[mask]
goals_clean = goals_per90[mask]

# Calculate correlation
corr_finishing, p_val_finishing = pearsonr(finishing_clean, goals_clean)

print(f"  Sample size: {len(finishing_clean)} forwards")
print(f"  Correlation: r = {corr_finishing:.3f}")
print(f"  P-value: p = {p_val_finishing:.6f}")
print(f"  Significance: {'***' if p_val_finishing < 0.001 else '**' if p_val_finishing < 0.01 else '*' if p_val_finishing < 0.05 else 'ns'}")

if corr_finishing > 0.8:
    print(f"  Interpretation: STRONG correlation - finishing rating is highly predictive")
elif corr_finishing > 0.6:
    print(f"  Interpretation: MODERATE correlation - finishing rating is meaningful")
else:
    print(f"  Interpretation: WEAK correlation - metric needs refinement")

# Validation 2: Chance creation vs actual assists (all positions)
print("\nValidation 2: Chance Creation Rating vs Actual Assists")
print("Scope: All positions")

all_with_creation = player_final.filter(
    pl.col('chance_creation_rating').is_not_null()
)

creation_ratings = all_with_creation['chance_creation_rating'].to_numpy()
assists_per90 = all_with_creation['assists_per90_actual'].to_numpy()

# Remove NaN values
mask = ~(np.isnan(creation_ratings) | np.isnan(assists_per90))
creation_clean = creation_ratings[mask]
assists_clean = assists_per90[mask]

# Calculate correlation
corr_creation, p_val_creation = pearsonr(creation_clean, assists_clean)

print(f"  Sample size: {len(creation_clean)} players")
print(f"  Correlation: r = {corr_creation:.3f}")
print(f"  P-value: p = {p_val_creation:.6f}")
print(f"  Significance: {'***' if p_val_creation < 0.001 else '**' if p_val_creation < 0.01 else '*' if p_val_creation < 0.05 else 'ns'}")

if corr_creation > 0.8:
    print(f"  Interpretation: STRONG correlation - chance creation rating is highly predictive")
elif corr_creation > 0.6:
    print(f"  Interpretation: MODERATE correlation - chance creation rating is meaningful")
else:
    print(f"  Interpretation: WEAK correlation - metric needs refinement")

# Summary
print("\nValidation summary:")
print(f"  Finishing vs Goals: r = {corr_finishing:.3f} (p < {p_val_finishing:.6f})")
print(f"  Chance Creation vs Assists: r = {corr_creation:.3f} (p < {p_val_creation:.6f})")

if corr_finishing > 0.8 and corr_creation > 0.8:
    print("\nConclusion: Both metrics show STRONG correlation with ground truth")
    print("Our rating system successfully captures real-world performance")
elif corr_finishing > 0.6 and corr_creation > 0.6:
    print("\nConclusion: Both metrics show MODERATE correlation with ground truth")
    print("Our rating system provides meaningful player evaluation")
else:
    print("\nConclusion: Metrics show varying correlation strength")
    print("Some metrics may need refinement")

# Save validation results
validation_results = {
    'metric': ['Finishing', 'Chance Creation'],
    'correlation': [corr_finishing, corr_creation],
    'p_value': [p_val_finishing, p_val_creation],
    'sample_size': [len(finishing_clean), len(creation_clean)]
}

validation_df = pl.DataFrame(validation_results)
validation_df.write_csv(PROCESSED_DIR / "validation_results.csv")

print(f"\nValidation results saved to: {PROCESSED_DIR / 'validation_results.csv'}")
print("\nMetric validation complete")

Validating metrics against ground truth statistics

Validation 1: Finishing Rating vs Actual Goals
Scope: Forwards only
  Sample size: 87 forwards
  Correlation: r = 0.857
  P-value: p = 0.000000
  Significance: ***
  Interpretation: STRONG correlation - finishing rating is highly predictive

Validation 2: Chance Creation Rating vs Actual Assists
Scope: All positions
  Sample size: 294 players
  Correlation: r = 0.829
  P-value: p = 0.000000
  Significance: ***
  Interpretation: STRONG correlation - chance creation rating is highly predictive

Validation summary:
  Finishing vs Goals: r = 0.857 (p < 0.000000)
  Chance Creation vs Assists: r = 0.829 (p < 0.000000)

Conclusion: Both metrics show STRONG correlation with ground truth
Our rating system successfully captures real-world performance

Validation results saved to: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/data/processed/validation_results.csv

Metric validation complete
